## Define notebook defaults and parameters

In [0]:
# define default parameters
# notebook user
notebook_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
notebook_user = notebook_user.split('@')[0].replace('.', '_')

# default catalog/schema
default_catalog = notebook_user + "_catalog"
default_schema = "kaggle_world_wide_importers"

# Parameters
dbutils.widgets.text("catalog_name", default_catalog)
dbutils.widgets.text("schema_name", default_schema)

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")

In [0]:
catalogs = [row.catalog for row in spark.sql("SHOW CATALOGS").collect()]
if catalog_name not in catalogs:
    raise ValueError(f"Catalog '{catalog_name}' does not exist.")

schemas = [row.databaseName for row in spark.sql(f"SHOW SCHEMAS IN {catalog_name}").collect()]
if schema_name not in schemas:
    raise ValueError(f"Schema '{schema_name}' does not exist in catalog '{catalog_name}'.")

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

## Set PK in all the tables

### application_cities PK

In [0]:
spark.sql("""
ALTER TABLE application_cities
ALTER COLUMN cityID SET NOT NULL
""")

spark.sql("""
ALTER TABLE application_cities
DROP CONSTRAINT IF EXISTS primary_key_cityid
""")

spark.sql("""
ALTER TABLE application_cities
ADD CONSTRAINT primary_key_cityid PRIMARY KEY (CityID)
""")

### application_stateprovinces PK

In [0]:
spark.sql("""
ALTER TABLE application_stateprovinces
ALTER COLUMN StateProvinceID SET NOT NULL
""")

spark.sql("""
ALTER TABLE application_stateprovinces
DROP CONSTRAINT IF EXISTS primary_key_stateprovinceid CASCADE
""")

spark.sql("""
ALTER TABLE application_stateprovinces
ADD CONSTRAINT primary_key_stateprovinceid PRIMARY KEY (StateProvinceID)
""")

### sales_orders PK

In [0]:
spark.sql("""
ALTER TABLE sales_orders
ALTER COLUMN OrderID SET NOT NULL
""")

spark.sql("""
ALTER TABLE sales_orders
DROP CONSTRAINT IF EXISTS primary_key_orderid CASCADE
""")

spark.sql("""
ALTER TABLE sales_orders
ADD CONSTRAINT primary_key_orderid PRIMARY KEY (OrderID)
""")

### sales_orderlines PK

In [0]:
spark.sql("""
ALTER TABLE sales_orderlines
ALTER COLUMN OrderID SET NOT NULL
""")

spark.sql("""
ALTER TABLE sales_orderlines
ALTER COLUMN OrderLineID SET NOT NULL
""")

spark.sql("""
ALTER TABLE amitabh_arora_catalog.kaggle_world_wide_importers.sales_orderlines
DROP CONSTRAINT IF EXISTS primary_key_orderid_orderlineid
""")

spark.sql("""
ALTER TABLE amitabh_arora_catalog.kaggle_world_wide_importers.sales_orderlines
ADD CONSTRAINT primary_key_orderid_orderlineid PRIMARY KEY (OrderID, OrderLineID)
""")

## Set FK for all the tables

### application_cities FK

In [0]:
spark.sql("""
ALTER TABLE application_cities
DROP CONSTRAINT IF EXISTS fk_stateprovinceid
""")

spark.sql(f"""
ALTER TABLE application_cities
ADD CONSTRAINT fk_stateprovinceid
FOREIGN KEY (StateProvinceID)
REFERENCES {catalog_name}.{schema_name}.application_stateprovinces (StateProvinceID)
""")

### sales_orderlines FK

In [0]:
spark.sql("""
ALTER TABLE sales_orderlines
DROP CONSTRAINT IF EXISTS foreign_key_orderid
""")

spark.sql("""
ALTER TABLE sales_orderlines
ADD CONSTRAINT foreign_key_orderid
FOREIGN KEY (OrderID) REFERENCES sales_orders(OrderID)
""")

# Testing